# Backpropagation through time in NumPy

A recurrent network reads a stream of symbols and, at every step, has to name the symbol it
saw four steps earlier. Nothing but its hidden state can carry a symbol across those steps,
so it can only learn the task if the gradient reaches back that far. You will write the
forward pass over a batch, the backward pass through time, and clipping by the global norm,
then train the network with them.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

V = 4      # symbols 0 to 3; inputs use 1 to 3, and target 0 means "nothing yet"


def softmax(logits):
    """Probabilities along the last axis, shifted by the row maximum so exp never overflows."""
    top = logits.max(axis=-1, keepdims=True)
    weights = np.exp(logits - top)
    return weights / weights.sum(axis=-1, keepdims=True)


def delay_batch(rng, B, T, lag=4):
    """B sequences of T random symbols from 1 to 3. The target at step t is the input at step t - lag."""
    xs = rng.integers(1, V, size=(B, T))
    ys = np.zeros_like(xs)
    ys[:, lag:] = xs[:, :-lag]
    return xs, ys


def init_params(rng, H):
    """Weights laid out as in nn.RNN and nn.Linear: W_xh is (H, V), W_hh (H, H), W_hy (V, H)."""
    return {
        "W_xh": rng.normal(0.0, 0.5, size=(H, V)),
        "W_hh": rng.normal(0.0, 0.5 / np.sqrt(H), size=(H, H)),
        "b_h": np.zeros(H),
        "W_hy": rng.normal(0.0, 0.5, size=(V, H)),
        "b_y": np.zeros(V),
    }


def numeric_grad(f, A, h=1e-6):
    """Central differences of f() in every entry of the array A. A is nudged in place and restored."""
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        old = A[idx]
        A[idx] = old + h
        up = f()
        A[idx] = old - h
        down = f()
        A[idx] = old
        G[idx] = (up - down) / (2 * h)
    return G


xs, ys = delay_batch(np.random.default_rng(0), 2, 10)
print("inputs: ", xs[0].tolist())
print("targets:", ys[0].tolist())

## Part 1 (coding task): The forward pass

Write `forward(params, xs, ys)` for integer arrays `xs` and `ys` of shape `(B, T)`. Each input
symbol enters as a one-hot vector $\mathbf{x}_t$ of length `V`, and with $\mathbf{h}_0 = \mathbf{0}$

$$
\mathbf{h}_t = \tanh\!\left(W_{xh}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b}_h\right), \qquad
\mathbf{p}_t = \operatorname{softmax}\!\left(W_{hy}\mathbf{h}_t + \mathbf{b}_y\right).
$$

The loss is the cross-entropy of the target symbol, $-\log p_t[y_t]$, averaged over all
$B \times T$ predictions. Return `(loss, cache)`, where `cache` holds whatever your backward
pass in Part 2 will need: the one-hot inputs, the states, the targets, and the probabilities
$\mathbf{p}_t$ under the key `ps`, shaped `(B, T, V)`, so the backward pass and the check in
Part 3 can find them there.

In [ ]:
def forward(params, xs, ys):
    """Mean cross-entropy over all B x T predictions, and a cache for backward."""
    # TODO
    raise NotImplementedError

In [ ]:
Hc = 3
pc = {"W_xh": 0.5 * np.cos(np.arange(Hc * V).reshape(Hc, V)),
      "W_hh": 0.4 * np.sin(np.arange(Hc * Hc).reshape(Hc, Hc) + 1.0),
      "b_h": np.array([0.1, -0.2, 0.05]),
      "W_hy": 0.7 * np.cos(np.arange(V * Hc).reshape(V, Hc) * 1.7),
      "b_y": np.array([0.0, 0.1, -0.1, 0.2])}
xs_c = np.array([[1, 3, 2, 2, 1], [2, 1, 1, 3, 3]])
ys_c = np.array([[0, 0, 1, 3, 2], [0, 0, 2, 1, 1]])

zero = {k: np.zeros_like(v) for k, v in pc.items()}
loss0 = float(forward(zero, xs_c, ys_c)[0])
for factor, what in ((10, "summed over all 10 predictions"), (5, "averaged over the batch only, not over the steps")):
    if abs(loss0 - factor * np.log(V)) < 1e-6:
        raise AssertionError(f"the loss is {factor} times too large: it looks {what}. Average over all B x T predictions")
assert abs(loss0 - np.log(V)) < 1e-9, f"with all parameters 0 every prediction is uniform, so the loss is ln 4 = 1.386, got {loss0:.4f}"
loss = float(forward(pc, xs_c, ys_c)[0])
want = 1.4597112674232489
assert abs(loss - want) < 1e-9, f"loss {loss:.6f}, expected {want:.6f}: check the recurrence, the one-hot inputs and which probability is picked"
print("Part 1 passed")

## Part 2 (coding task): The backward pass through time

Write `backward(params, cache)`. It returns a dict with the same keys as `params`, holding
the gradient of the loss from Part 1 with respect to each parameter, shaped like it.

The check compares every entry with central differences of your `forward`.

In [ ]:
def backward(params, cache):
    """Gradients of the mean cross-entropy, one array per parameter."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
p = init_params(rng, 5)
p["b_h"] = rng.normal(0.0, 0.3, size=5)
p["b_y"] = rng.normal(0.0, 0.3, size=V)
xs_g, ys_g = delay_batch(rng, 3, 7)
grads = backward(p, forward(p, xs_g, ys_g)[1])
assert set(grads) == set(p), f"expected the keys {sorted(p)}, got {sorted(grads)}"
for name in ("W_hy", "b_y", "W_hh", "W_xh", "b_h"):
    assert np.shape(grads[name]) == p[name].shape, f"{name}: gradient shape {np.shape(grads[name])}, parameter shape {p[name].shape}"
    num = numeric_grad(lambda: forward(p, xs_g, ys_g)[0], p[name])
    if not np.allclose(grads[name], num, atol=1e-7):
        if name in ("W_hy", "b_y"):
            raise AssertionError(f"{name} does not match the numeric gradient: look at the error at the softmax output "
                                 "and what the mean divides by")
        raise AssertionError(f"the output layer is right and {name} is not: look at everything that reaches h_t, "
                             "from its own output and from step t + 1, and at the tanh slope")
print("Part 2 passed")

## Part 3 (coding task): Clipping by the global norm

Write `clip_by_global_norm(grads, tau)`, which returns `(clipped, norm)`. `norm` is the length
of the whole gradient, all tensors together, and `clipped` is a new dict holding the gradients
scaled by $\tau / \text{norm}$ when `norm` exceeds $\tau$, or unchanged when it does not. Leave
the arrays passed in unchanged.

The check then trains the network on the delay task with your three functions.

In [ ]:
def clip_by_global_norm(grads, tau):
    """(clipped copy of grads, length of the whole gradient before clipping)."""
    # TODO
    raise NotImplementedError

In [ ]:
g = {"a": np.array([[1.0, -2.0], [2.0, 0.0]]),   # length 3
     "b": np.array([0.0, 6.0]),                  # length 6
     "c": np.array([2.0])}                       # length 2
before = {k: v.copy() for k, v in g.items()}
clipped, norm = clip_by_global_norm(g, 3.5)
assert all(np.array_equal(g[k], before[k]) for k in g), "the gradients passed in were changed: return scaled copies"
assert abs(norm - 7.0) < 1e-9, f"norm {norm}: the length of all the entries together is 7"
if np.allclose(clipped["c"], 2.0):
    raise AssertionError("the length-2 tensor was left alone: one factor, tau / 7, scales every tensor, including the short ones")
assert np.allclose(clipped["b"], [0.0, 3.0]) and np.allclose(clipped["c"], [1.0]) and np.allclose(clipped["a"], 0.5 * before["a"]), \
    "with tau = 3.5 and a total length of 7, every entry should be halved"
same, norm = clip_by_global_norm(g, 10.0)
assert abs(norm - 7.0) < 1e-9 and all(np.allclose(same[k], before[k]) for k in g), "below the threshold nothing should change"

# Train with the three functions: 400 steps of SGD on batches of 32 sequences of 16 steps.
rng = np.random.default_rng(2)
params = init_params(rng, 16)
norms = []
for step in range(400):
    xs, ys = delay_batch(rng, 32, 16)
    loss, cache = forward(params, xs, ys)
    grads, n = clip_by_global_norm(backward(params, cache), 1.0)
    norms.append(n)
    for k in params:
        params[k] -= 1.0 * grads[k]
xs, ys = delay_batch(np.random.default_rng(3), 500, 16)
loss, cache = forward(params, xs, ys)
acc = float(np.mean(cache["ps"].argmax(axis=2)[:, 4:] == ys[:, 4:]))
norms = np.array(norms)
print(f"steps clipped: {np.sum(norms > 1.0)} of 400, largest gradient length {norms.max():.2f} at step {norms.argmax()}")
print(f"held-out loss {loss:.3f}, accuracy on the delayed symbols {acc:.3f}")
assert acc > 0.95, "the network did not learn the delay with your three functions"
print("Part 3 passed")

## Solution

### Part 1: The forward pass

In [ ]:
def forward(params, xs, ys):
    """Mean cross-entropy over all B x T predictions, and a cache for backward."""
    B, T = xs.shape
    X = np.eye(V)[xs]                                   # (B, T, V) one-hot inputs
    H = params["W_hh"].shape[0]
    hs = np.zeros((B, T + 1, H))                        # hs[:, t] is h_t, with hs[:, 0] = h_0 = 0
    ps = np.zeros((B, T, V))
    for t in range(T):
        hs[:, t + 1] = np.tanh(X[:, t] @ params["W_xh"].T + hs[:, t] @ params["W_hh"].T + params["b_h"])
        ps[:, t] = softmax(hs[:, t + 1] @ params["W_hy"].T + params["b_y"])
    picked = np.take_along_axis(ps, ys[..., None], axis=2)[..., 0]
    return -np.mean(np.log(picked)), {"X": X, "hs": hs, "ps": ps, "ys": ys}

**Why it works.** The one-hot input picks one column of $W_{xh}$, so `np.eye(V)[xs]` turns the
whole integer array into inputs at once. Keeping every state, $\mathbf{h}_0$ included, costs memory that
grows with $T$, the cost the lesson gives for full BPTT, and the backward pass needs all of
them. The mean over all $B \times T$ predictions keeps the size of the gradient independent
of the batch size and the sequence length, so one learning rate works for both.

### Part 2: The backward pass through time

In [ ]:
def backward(params, cache):
    """Gradients of the mean cross-entropy, one array per parameter."""
    X, hs, ps, ys = cache["X"], cache["hs"], cache["ps"], cache["ys"]
    B, T, _ = ps.shape
    grads = {k: np.zeros_like(v) for k, v in params.items()}
    dlogits = ps.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], ys] -= 1.0
    dlogits /= B * T                                    # the loss is a mean over B x T terms
    dh_next = np.zeros((B, params["W_hh"].shape[0]))    # what h_t receives from step t + 1
    for t in reversed(range(T)):
        grads["W_hy"] += dlogits[:, t].T @ hs[:, t + 1]
        grads["b_y"] += dlogits[:, t].sum(axis=0)
        dh = dlogits[:, t] @ params["W_hy"] + dh_next   # own output plus the next state
        dz = dh * (1.0 - hs[:, t + 1] ** 2)             # through the tanh
        grads["W_xh"] += dz.T @ X[:, t]
        grads["W_hh"] += dz.T @ hs[:, t]                # paired with h_(t-1)
        grads["b_h"] += dz.sum(axis=0)
        dh_next = dz @ params["W_hh"]
    return grads

**Why it works.** Each hidden state reaches the loss along two routes, through its own
output and through the next state, so the backward loop carries `dh_next` from step $t + 1$ and
adds it to what comes down from the output at step $t$. Both pass through the tanh slope,
$1 - \mathbf{h}_t^2$, before reaching the weights. Each weight is used at every step, so its
gradient is a sum of one outer product per step, and the one for $W_{hh}$ pairs the error at step
$t$ with $\mathbf{h}_{t-1}$, the state it multiplied. Leaving out `dh_next` still gives the right output layer
and plausible-looking numbers everywhere else, which is why every entry is compared with
central differences.

### Part 3: Clipping by the global norm

In [ ]:
def clip_by_global_norm(grads, tau):
    """(clipped copy of grads, length of the whole gradient before clipping)."""
    norm = float(np.sqrt(sum(np.sum(g ** 2) for g in grads.values())))
    scale = tau / norm if norm > tau else 1.0
    return {k: g * scale for k, g in grads.items()}, norm

**Why it works.** The length of the whole gradient is the square root of the sum of the
squares of every entry of every tensor, and one factor scales them all, so the update keeps the
direction of the full gradient. Clipping each tensor on its own, or capping entries, would turn
it. In the training run every clipped step comes in the first 90 or so, while the network is
learning to hold a symbol for four steps and the loss falls from about 1.5 to under 0.1. After that
the gradient stays far below 1. Those early spikes are what clipping is for: one long step at
that moment can undo what has been learned.